# A2A (Agent2Agent) + mycontext SynthesisBuilder

Simple integration: **SynthesisBuilder** template as A2A agent system prompt → verify templates work.

**Template:** `synthesis_builder` — synthesizes multiple sources into coherent summary

**Prerequisites:**
```bash
pip install "mycontext-ai[openai]" "python-a2a[openai]"
```
Set `OPENAI_API_KEY`.

In [1]:
from mycontext.intelligence import get_pattern_class

SynthesisBuilder = get_pattern_class("synthesis_builder")
sources = ["Report A: 20% growth", "Report B: strong retention", "Report C: new market entry"]

ctx = SynthesisBuilder().build_context(
    sources=sources,
    goal="Executive summary of quarterly performance"
)
system_prompt = ctx.assemble()
print("Context built. System prompt length:", len(system_prompt), "chars")

Context built. System prompt length: 2088 chars


In [2]:
try:
    import os
    import threading
    import time

    from python_a2a import A2AClient, Message, MessageRole, OpenAIA2AServer, TextContent, run_server
except ImportError:
    raise ImportError('pip install "python-a2a[openai]"')

server_ready = threading.Event()
PORT = 5050

def run_a2a_server():
    agent = OpenAIA2AServer(
        api_key=os.environ.get("OPENAI_API_KEY"),
        model="gpt-4o-mini",
        system_prompt=system_prompt,
    )
    server_ready.set()
    run_server(agent, host="127.0.0.1", port=PORT)

t = threading.Thread(target=run_a2a_server, daemon=True)
t.start()
server_ready.wait(timeout=5)
time.sleep(1)

client = A2AClient(f"http://127.0.0.1:{PORT}/a2a")
msg = Message(content=TextContent(text=f"Synthesize these sources: {sources}"), role=MessageRole.USER)
resp = client.send_message(msg)
print(resp.content.text)
print("\n--- Template works with A2A ---")

ImportError: pip install "python-a2a[openai]"